# MovieLens 数据探索
## CS5100 Final Project - Data Exploration

这个notebook用于探索MovieLens 100K数据集，包括：
- 加载数据
- 基本统计信息
- 数据可视化
- 数据质量检查

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# 添加src目录到路径
sys.path.append('../src')
from data_loader import MovieLensLoader

# 设置绘图样式
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports successful!")

## 1. 加载数据

In [ ]:
# 创建数据加载器
loader = MovieLensLoader(data_path='../data/raw/ml-100k')

# 加载所有数据
ratings, movies, users = loader.load_all()

print("\n数据加载完成！")

In [ ]:
# 查看评分数据的前几行
print("评分数据示例：")
ratings.head(10)

In [ ]:
# 查看电影数据的前几行
print("电影数据示例：")
movies[['item_id', 'title', 'Action', 'Comedy', 'Drama']].head(10)

## 2. 数据集统计信息

In [ ]:
# 打印详细统计信息
loader.print_stats()

In [ ]:
# 评分数据的描述性统计
print("评分分布统计：")
ratings['rating'].describe()

## 3. 数据可视化

### 3.1 评分分布

In [ ]:
# 评分分布直方图
plt.figure(figsize=(10, 6))
ratings['rating'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Rating Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Rating', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)

# 添加百分比标签
total = len(ratings)
for i, v in enumerate(ratings['rating'].value_counts().sort_index()):
    plt.text(i, v + 1000, f'{v/total*100:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/rating_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n评分分布：")
print(ratings['rating'].value_counts().sort_index())

### 3.2 用户活跃度分布

In [ ]:
# 每个用户的评分数量
ratings_per_user = ratings.groupby('user_id').size()

plt.figure(figsize=(12, 5))

# 直方图
plt.subplot(1, 2, 1)
plt.hist(ratings_per_user, bins=50, color='coral', edgecolor='black', alpha=0.7)
plt.title('User Activity Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Ratings per User', fontsize=11)
plt.ylabel('Number of Users', fontsize=11)
plt.axvline(ratings_per_user.mean(), color='red', linestyle='--', 
            label=f'Mean: {ratings_per_user.mean():.1f}')
plt.legend()
plt.grid(alpha=0.3)

# 箱线图
plt.subplot(1, 2, 2)
plt.boxplot(ratings_per_user, vert=True)
plt.title('User Activity Box Plot', fontsize=14, fontweight='bold')
plt.ylabel('Number of Ratings', fontsize=11)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/user_activity.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n用户活跃度统计：")
print(f"平均每用户评分数: {ratings_per_user.mean():.1f}")
print(f"中位数: {ratings_per_user.median():.1f}")
print(f"最少: {ratings_per_user.min()}")
print(f"最多: {ratings_per_user.max()}")

### 3.3 电影热门度分布

In [ ]:
# 每部电影的评分数量
ratings_per_movie = ratings.groupby('item_id').size()

plt.figure(figsize=(12, 5))

# 直方图
plt.subplot(1, 2, 1)
plt.hist(ratings_per_movie, bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
plt.title('Movie Popularity Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Ratings per Movie', fontsize=11)
plt.ylabel('Number of Movies', fontsize=11)
plt.axvline(ratings_per_movie.mean(), color='red', linestyle='--',
            label=f'Mean: {ratings_per_movie.mean():.1f}')
plt.legend()
plt.grid(alpha=0.3)

# 对数尺度
plt.subplot(1, 2, 2)
plt.hist(ratings_per_movie, bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
plt.title('Movie Popularity (Log Scale)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Ratings per Movie', fontsize=11)
plt.ylabel('Number of Movies', fontsize=11)
plt.yscale('log')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/movie_popularity.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n电影热门度统计：")
print(f"平均每电影评分数: {ratings_per_movie.mean():.1f}")
print(f"中位数: {ratings_per_movie.median():.1f}")
print(f"最少: {ratings_per_movie.min()}")
print(f"最多: {ratings_per_movie.max()}")

# 冷门电影（评分少于5次）
cold_movies = (ratings_per_movie < 5).sum()
print(f"\n冷门电影（<5个评分）: {cold_movies} ({cold_movies/len(ratings_per_movie)*100:.1f}%)")

### 3.4 最受欢迎的电影

In [ ]:
# 计算每部电影的平均评分和评分数量
movie_stats = ratings.groupby('item_id').agg({
    'rating': ['mean', 'count']
}).reset_index()
movie_stats.columns = ['item_id', 'avg_rating', 'num_ratings']

# 合并电影标题
movie_stats = movie_stats.merge(movies[['item_id', 'title']], on='item_id')

# 只考虑至少有50个评分的电影
popular_movies = movie_stats[movie_stats['num_ratings'] >= 50]

# Top 10最高评分电影
top_rated = popular_movies.nlargest(10, 'avg_rating')

print("\n🏆 Top 10 最高评分电影（至少50个评分）：")
print(top_rated[['title', 'avg_rating', 'num_ratings']].to_string(index=False))

In [ ]:
# Top 10评分最多的电影
most_rated = movie_stats.nlargest(10, 'num_ratings')

print("\n📊 Top 10 评分最多的电影：")
print(most_rated[['title', 'avg_rating', 'num_ratings']].to_string(index=False))

### 3.5 电影类型分布

In [ ]:
# 统计每种类型的电影数量
genre_columns = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy',
                'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 
                'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 
                'Thriller', 'War', 'Western']

genre_counts = movies[genre_columns].sum().sort_values(ascending=True)

# 水平条形图
plt.figure(figsize=(10, 8))
genre_counts.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Movie Genre Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Number of Movies', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.grid(axis='x', alpha=0.3)

# 添加数值标签
for i, v in enumerate(genre_counts):
    plt.text(v + 5, i, str(int(v)), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../results/figures/genre_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n电影类型统计：")
print(genre_counts.sort_values(ascending=False))

### 3.6 数据稀疏性可视化

In [ ]:
# 创建评分矩阵（抽样显示）
# 只显示前100个用户和前100部电影
sample_ratings = ratings[
    (ratings['user_id'] <= 100) & 
    (ratings['item_id'] <= 100)
]

sample_matrix = sample_ratings.pivot(
    index='user_id',
    columns='item_id',
    values='rating'
)

# 可视化稀疏矩阵
plt.figure(figsize=(12, 10))
plt.spy(sample_matrix.notna(), markersize=1, aspect='auto')
plt.title('Rating Matrix Sparsity (First 100 users × 100 movies)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Movie ID', fontsize=11)
plt.ylabel('User ID', fontsize=11)
plt.tight_layout()
plt.savefig('../results/figures/sparsity_pattern.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n样本矩阵稀疏度: {sample_matrix.isna().sum().sum() / sample_matrix.size * 100:.1f}%")

## 4. 数据质量检查

In [ ]:
print("数据质量检查：\n")

# 检查缺失值
print("1. 缺失值检查")
print(f"   评分数据缺失值: {ratings.isna().sum().sum()}")
print(f"   电影数据缺失值: {movies.isna().sum().sum()}")
print(f"   用户数据缺失值: {users.isna().sum().sum()}")

# 检查重复值
print("\n2. 重复值检查")
duplicates = ratings.duplicated(subset=['user_id', 'item_id']).sum()
print(f"   重复的(user, item)对: {duplicates}")

# 检查评分范围
print("\n3. 评分范围检查")
invalid_ratings = ((ratings['rating'] < 1) | (ratings['rating'] > 5)).sum()
print(f"   不在1-5范围内的评分: {invalid_ratings}")

# 检查ID连续性
print("\n4. ID连续性检查")
missing_users = set(range(1, ratings['user_id'].max() + 1)) - set(ratings['user_id'].unique())
missing_movies = set(range(1, ratings['item_id'].max() + 1)) - set(ratings['item_id'].unique())
print(f"   缺失的用户ID: {len(missing_users)}")
print(f"   缺失的电影ID: {len(missing_movies)}")

print("\n✅ 数据质量检查完成！")

## 5. 保存处理后的数据

In [ ]:
# 划分训练集和测试集
train_data, test_data = loader.split_train_test(test_size=0.2, random_state=42)

# 保存到processed文件夹
train_data.to_csv('../data/processed/train_data.csv', index=False)
test_data.to_csv('../data/processed/test_data.csv', index=False)

print("\n✅ 训练集和测试集已保存到 data/processed/")

## 总结

通过这个数据探索，我们发现：

1. **数据规模**: 100,000条评分，943个用户，1,682部电影
2. **评分分布**: 评分集中在3-4星，呈现正态分布
3. **数据稀疏性**: 约93.7%的稀疏度，说明大部分用户只评价了少数电影
4. **用户活跃度**: 差异较大，有些用户评分很多，有些很少
5. **电影热门度**: 长尾分布，少数电影很热门，大多数电影评分较少
6. **类型分布**: Drama和Comedy是最常见的类型
7. **数据质量**: 无缺失值，无重复，评分都在有效范围内

这些发现将帮助我们：
- 理解baseline推荐的表现
- 处理冷启动问题
- 设计合适的推荐算法